<a href="https://colab.research.google.com/github/molluIdontknow/ELE_Algorithm/blob/%EA%B9%80%EB%AF%BC%EC%A0%9C/ELEalgo_MLP_improve.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ELEalgo — MLP 개선 실험 노트북

`ELEalgo_model_eval`에서 MLP(25 feature) 5-fold RMSE 0.69가 나왔다. 여기서는 **구조(N→16→8→1)는 FPGA 합의라 고정**하고,
141행짜리 작은 데이터에서 실제로 효과가 있는 개선 기법이 뭔지 **같은 fold × 3 seed**로 하나씩 검증한다.

| 그룹 | 실험 | 추론(FPGA)에 영향? |
|---|---|---|
| A. 데이터 증강 | 입력 가우시안 노이즈 / 같은 셀 RPT 사이 보간 | 없음 (학습 때만) |
| B. 손실·규제 | Huber loss / Dropout / weight decay↑ | 없음 |
| C. 최적화 | Cosine LR / 긴 patience | 없음 |
| D. 입력 처리 | RobustScaler / log 변환 | **있음** (펌웨어 정규화 식 바뀜) |
| E. Feature 축소 | 중복 제거 / ccct 제거 / 배포 가능 세트 | **있음** (입력 차원↓ → MAC↓) |
| F. 구조 | hidden 8-4 / 16-8 / 32-16 | **있음** |
| G. 앙상블 | seed 5개 평균 | **있음** (×5 비용) |
| H. QAT | int8 weight 양자화 인지 학습 | 없음 (양자화 손실↓) |

마지막에 효과 있는 것만 합쳐서 **최종 config**를 만들고, 기본 config와 **10 seed × holdout**으로 비교한다.

**실행 시간**: CPU 약 8~12분.


# §0. 환경 / 데이터 / 공통 함수

In [ ]:
import os, json, time, random, warnings, copy
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib as mpl
import torch, torch.nn as nn, torch.optim as optim
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")
mpl.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
torch.set_num_threads(2)

def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)

# ── 데이터 ──────────────────────────────────────────────────────────────────
DRIVE_PATH = "/content/drive/MyDrive/merged_features_25C.csv"; LOCAL_PATH = "merged_features_25C.csv"
if os.path.exists(DRIVE_PATH): CSV = DRIVE_PATH
elif os.path.exists(LOCAL_PATH): CSV = LOCAL_PATH
else:
    from google.colab import drive; drive.mount("/content/drive"); CSV = DRIVE_PATH

META, TARGET, GROUP = ["cell_global_id", "rpt_id", "aging_condition"], "soh_label_pct", "cell_global_id"
df = pd.read_csv(CSV, encoding="utf-8-sig")
df["cond"] = df["aging_condition"].str.replace(r"\s*#\d+$", "", regex=True)
ALL_FEATURES = [c for c in df.columns if c not in META + [TARGET, "cond"]]

X_all = df[ALL_FEATURES].values.astype(np.float32)
y_all = df[TARGET].values.astype(np.float32)
g_all = df[GROUP].values
r_all = df["rpt_id"].values

# holdout / dev 분할은 eval 노트북과 동일 (seed 42) → 결과 비교 가능
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
dev_idx, test_idx = next(gss.split(X_all, y_all, groups=g_all))
X_dev, y_dev, g_dev, r_dev = X_all[dev_idx], y_all[dev_idx], g_all[dev_idx], r_all[dev_idx]
X_test, y_test, g_test = X_all[test_idx], y_all[test_idx], g_all[test_idx]
FOLDS = list(GroupKFold(n_splits=5).split(X_dev, y_dev, groups=g_dev))
print(f"dev {len(X_dev)} rows / {len(np.unique(g_dev))} cells,  test {len(X_test)} rows / {len(np.unique(g_test))} cells,  features {len(ALL_FEATURES)}")

def fidx(names): return [ALL_FEATURES.index(n) for n in names]   # feature 이름 → 열 인덱스
def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))

## 0-1. 확장된 학습 함수

`train_mlp`에 실험용 옵션을 전부 넣었다. 기본값이면 eval 노트북과 완전히 같은 동작.

| cfg 키 | 기본 | 뜻 |
|---|---|---|
| `hidden` | (16, 8) | hidden layer 크기 |
| `dropout` | 0 | hidden 뒤 dropout 확률 |
| `loss` | "mse" | "mse" / "huber" (δ=1.0, 스케일된 타깃 기준 → 실제 10 %SOH) |
| `noise_std` | 0 | 매 batch 입력에 N(0, σ²) 노이즈 (표준화 공간) |
| `interp_aug` | 0 | 같은 셀의 인접 RPT 쌍을 보간해서 매 epoch 추가할 가짜 샘플 수 |
| `scheduler` | "plateau" | "plateau" / "cosine" |
| `scaler` | "standard" | "standard" / "robust" |
| `log_cols` | [] | log 변환할 열 인덱스 (스케일링 전에 적용, 양수 feature만) |
| `qat` | False | forward 때 weight를 int8로 fake-quantize (straight-through) |


In [ ]:
BASE_CFG = {
    "batch_size": 16, "epochs": 2000, "learning_rate": 3e-3, "weight_decay": 1e-4,
    "patience": 100, "lr_patience": 30,
    "hidden": (16, 8), "dropout": 0.0, "loss": "mse",
    "noise_std": 0.0, "interp_aug": 0,
    "scheduler": "plateau", "scaler": "standard", "log_cols": [],
    "qat": False,
}
Y_OFFSET, Y_SCALE = 90.0, 10.0
def y_enc(y): return (y - Y_OFFSET) / Y_SCALE
def y_dec(z): return z * Y_SCALE + Y_OFFSET

# ── 입력 전처리: log 변환 + scaler 를 하나로 묶음 (펌웨어 전달 시 둘 다 상수로 나감) ──
class Preproc:
    def __init__(self, scaler="standard", log_cols=()):
        self.log_cols = list(log_cols)
        self.sc = StandardScaler() if scaler == "standard" else RobustScaler()
    def _log(self, X):
        X = X.astype(np.float64).copy()
        if self.log_cols: X[:, self.log_cols] = np.log(np.clip(X[:, self.log_cols], 1e-9, None))
        return X
    def fit(self, X): self.sc.fit(self._log(X)); return self
    def transform(self, X): return self.sc.transform(self._log(X)).astype(np.float32)
    def params(self):   # 펌웨어용 상수
        if isinstance(self.sc, StandardScaler): return {"center": self.sc.mean_.tolist(), "scale": self.sc.scale_.tolist()}
        return {"center": self.sc.center_.tolist(), "scale": self.sc.scale_.tolist()}

# ── fake-quant (QAT): forward는 int8로 반올림한 weight, backward는 그대로 통과 ──
class FakeQuant(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w):
        s = w.abs().max() / 127.0 + 1e-12
        return torch.clamp(torch.round(w / s), -127, 127) * s
    @staticmethod
    def backward(ctx, g): return g

class QLinear(nn.Linear):
    qat = False
    def forward(self, x):
        w = FakeQuant.apply(self.weight) if self.qat else self.weight
        return nn.functional.linear(x, w, self.bias)

class MLP(nn.Module):
    def __init__(self, d_in, hidden=(16, 8), dropout=0.0, qat=False):
        super().__init__()
        layers, d = [], d_in
        for h in hidden:
            lin = QLinear(d, h); lin.qat = qat
            layers += [lin, nn.ReLU()] + ([nn.Dropout(dropout)] if dropout > 0 else [])
            d = h
        out = QLinear(d, 1); out.qat = qat
        layers.append(out)
        self.network = nn.Sequential(*layers)
    def forward(self, x): return self.network(x)
    def set_qat(self, flag):
        for m in self.network:
            if isinstance(m, QLinear): m.qat = flag

def count_params(m): return sum(p.numel() for p in m.parameters())

# ── 셀 내부 보간 증강: 같은 셀에서 RPT 순서상 인접한 두 행을 α로 섞음 ──────────
def interp_pairs(g, r):
    # 반환: (i, j) 인덱스 쌍 — 같은 셀, rpt 순서 인접
    pairs = []
    for cid in np.unique(g):
        idx = np.where(g == cid)[0]; idx = idx[np.argsort(r[idx])]
        pairs += list(zip(idx[:-1], idx[1:]))
    return np.array(pairs)

def train_mlp(X_tr, y_tr, X_va, y_va, g_tr=None, r_tr=None, cfg=None, verbose=False):
    cfg = {**BASE_CFG, **(cfg or {})}
    pp = Preproc(cfg["scaler"], cfg["log_cols"]).fit(X_tr)
    Xtr = torch.tensor(pp.transform(X_tr)); ytr = torch.tensor(y_enc(y_tr), dtype=torch.float32).unsqueeze(1)
    Xva = torch.tensor(pp.transform(X_va)); yva = torch.tensor(y_enc(y_va), dtype=torch.float32).unsqueeze(1)
    pairs = interp_pairs(g_tr, r_tr) if (cfg["interp_aug"] > 0 and g_tr is not None) else None

    model = MLP(X_tr.shape[1], cfg["hidden"], cfg["dropout"], cfg["qat"])
    crit = nn.MSELoss() if cfg["loss"] == "mse" else nn.HuberLoss(delta=1.0)
    opt = optim.Adam(model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
    if cfg["scheduler"] == "plateau":
        sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=cfg["lr_patience"])
    else:
        sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=200, T_mult=1, eta_min=cfg["learning_rate"] * 0.01)

    best, best_state, best_ep, wait = float("inf"), None, 0, 0
    hist = {"train": [], "val": []}
    n = len(Xtr)
    for ep in range(1, cfg["epochs"] + 1):
        # ---- epoch 데이터 구성 (원본 + 보간 증강) ----
        if pairs is not None:
            k = cfg["interp_aug"]
            sel = pairs[np.random.randint(len(pairs), size=k)]
            a = torch.rand(k, 1)
            Xa = a * Xtr[sel[:, 0]] + (1 - a) * Xtr[sel[:, 1]]
            ya = a * ytr[sel[:, 0]] + (1 - a) * ytr[sel[:, 1]]
            Xe, ye = torch.cat([Xtr, Xa]), torch.cat([ytr, ya])
        else:
            Xe, ye = Xtr, ytr
        perm = torch.randperm(len(Xe))
        model.train(); tl = 0.0; nb = 0
        for s in range(0, len(Xe), cfg["batch_size"]):
            b = perm[s:s + cfg["batch_size"]]
            xb = Xe[b]
            if cfg["noise_std"] > 0: xb = xb + cfg["noise_std"] * torch.randn_like(xb)   # 입력 노이즈 증강
            opt.zero_grad(); loss = crit(model(xb), ye[b]); loss.backward(); opt.step()
            tl += loss.item(); nb += 1
        model.eval()
        with torch.no_grad(): vl = nn.functional.mse_loss(model(Xva), yva).item()
        sched.step(vl) if cfg["scheduler"] == "plateau" else sched.step()
        hist["train"].append(tl / nb); hist["val"].append(vl)
        if vl < best - 1e-6:
            best, best_ep, wait = vl, ep, 0
            best_state = copy.deepcopy(model.state_dict())
        else: wait += 1
        if verbose and ep % 100 == 0: print(f"  ep {ep:4d} train {tl/nb:.4f} val {vl:.4f}")
        if wait >= cfg["patience"]: break
    model.load_state_dict(best_state)
    return model, pp, hist, best_ep

def predict(model, pp, X):
    model.eval()
    with torch.no_grad(): return y_dec(model(torch.tensor(pp.transform(X))).numpy().flatten())

def metrics(y, p):
    return {"RMSE": rmse(y, p), "MAE": float(mean_absolute_error(y, p)), "R2": float(r2_score(y, p)),
            "ErrRate_%": float(np.mean(np.abs(y - p) / y) * 100), "MaxAbsErr": float(np.max(np.abs(y - p)))}

## 0-2. 실험 평가 함수

`run_exp(name, cfg, feats, seeds)` → 5-fold × seeds 학습 → fold-RMSE 평균/표준편차 + OOF 지표 + 학습 시간.
결과는 전역 `RESULTS`에 쌓이고 마지막에 표/그래프로 비교한다.


In [ ]:
RESULTS = []
OOF_STORE = {}
SEEDS = (0, 1, 2)

def run_exp(name, cfg=None, feats=None, seeds=SEEDS, group=""):
    cfg = cfg or {}
    fi = fidx(feats) if feats is not None else list(range(len(ALL_FEATURES)))
    fold_rmse, oof = [], np.zeros((len(seeds), len(y_dev)))
    t0 = time.perf_counter(); eps = []
    for si, s in enumerate(seeds):
        for k, (tr, va) in enumerate(FOLDS, 1):
            set_seed(100 * s + k)
            m, pp, _, ep = train_mlp(X_dev[tr][:, fi], y_dev[tr], X_dev[va][:, fi], y_dev[va],
                                     g_tr=g_dev[tr], r_tr=r_dev[tr], cfg=cfg)
            p = predict(m, pp, X_dev[va][:, fi]); oof[si, va] = p
            fold_rmse.append(rmse(y_dev[va], p)); eps.append(ep)
    oof_mean = oof.mean(axis=0)                       # seed 평균 예측 (참고용)
    mt = metrics(y_dev, oof_mean)
    row = {"group": group, "exp": name, "n_feat": len(fi),
           "fold_RMSE_mean": np.mean(fold_rmse), "fold_RMSE_std": np.std(fold_rmse),
           "OOF_RMSE(seed-avg)": mt["RMSE"], "OOF_MAE": mt["MAE"], "OOF_R2": mt["R2"], "OOF_ErrRate_%": mt["ErrRate_%"],
           "best_ep": np.mean(eps), "sec": time.perf_counter() - t0}
    RESULTS.append(row); OOF_STORE[name] = oof
    print(f"[{group:>2}] {name:34s} feat {len(fi):2d}  fold RMSE {row['fold_RMSE_mean']:.3f} ± {row['fold_RMSE_std']:.3f}   OOF {mt['RMSE']:.3f}   ({row['sec']:.0f}s)")
    return row

def results_df():
    return pd.DataFrame(RESULTS).set_index("exp")

# §1. Baseline

eval 노트북과 같은 설정. 여기 숫자가 기준선. (seed 3개 평균이라 eval의 0.69와 약간 다를 수 있음)


In [ ]:
base = run_exp("baseline (all25, default)", {}, group="0")

# §2. A — 데이터 증강 (학습 때만, FPGA 무관)

- **입력 노이즈**: 표준화된 입력에 N(0, σ²). σ=0.05는 "측정 노이즈 5%" 정도. 과적합을 줄이고 보드 측정 노이즈에도 강해짐.
- **셀 내 보간**: 같은 셀의 RPT k와 k+1 사이를 선형보간한 가짜 샘플. 열화는 연속적이니 물리적으로 그럴듯한 증강. (다른 셀끼리 섞는 mixup은 안 함 — 조건이 다른 셀을 섞으면 비물리적)


In [ ]:
for s in [0.03, 0.05, 0.1]:
    run_exp(f"A: noise σ={s}", {"noise_std": s}, group="A")
for k in [32, 64, 128]:
    run_exp(f"A: interp aug +{k}/epoch", {"interp_aug": k}, group="A")
run_exp("A: noise 0.05 + interp 64", {"noise_std": 0.05, "interp_aug": 64}, group="A")

# §3. B — 손실 함수 / 규제

- **Huber**: 큰 잔차(4.5V 조건 같은 이상 셀)에 덜 끌려감. δ=1.0 (스케일 공간) = 10 %SOH 이상부터 L1.
- **Dropout**: hidden 16·8에 0.1~0.2. 뉴런이 적어서 오히려 해로울 수 있음 → 확인 목적.
- **weight decay↑**: 지난 튜닝에서 효과 없었지만 25 feature에선 다를 수 있음.


In [ ]:
run_exp("B: huber loss", {"loss": "huber"}, group="B")
for d in [0.1, 0.2]:
    run_exp(f"B: dropout {d}", {"dropout": d}, group="B")
for wd in [1e-3, 1e-2]:
    run_exp(f"B: weight_decay {wd:g}", {"weight_decay": wd}, group="B")

# §4. C — 최적화 스케줄

- **Cosine warm restart** (T=200): lr을 주기적으로 올렸다 내림 → 얕은 local minimum 탈출.
- **patience 200 + lr 1e-3**: 더 천천히, 더 오래.


In [ ]:
run_exp("C: cosine LR", {"scheduler": "cosine", "patience": 300}, group="C")
run_exp("C: lr 1e-3, patience 200", {"learning_rate": 1e-3, "patience": 200}, group="C")

# §5. D — 입력 처리 (펌웨어 정규화 식이 바뀜)

- **RobustScaler**: 평균/표준편차 대신 중앙값/IQR. 이상치(4.5V 셀)가 스케일을 망치는 걸 막음.
- **log 변환**: DCIR·시간 계열은 오른쪽 꼬리 분포 → log 취하면 대칭에 가까워짐. 펌웨어에서 log 1회 필요 (룩업테이블 가능).


In [ ]:
LOG_CANDIDATES = [f for f in ALL_FEATURES if f.startswith("dcir") or f.startswith("deltat") or f in ("ccct", "cvct")]
print("log 변환 후보:", LOG_CANDIDATES)
run_exp("D: robust scaler", {"scaler": "robust"}, group="D")
run_exp("D: log(DCIR, time)", {"log_cols": fidx(LOG_CANDIDATES)}, group="D")
run_exp("D: log + robust", {"log_cols": fidx(LOG_CANDIDATES), "scaler": "robust"}, group="D")

# §6. E — Feature 축소 (입력 차원↓ = MAC↓, 과적합↓)

eval 노트북 §8 결과를 근거로 세트 정의:

| 세트 | 내용 |
|---|---|
| `dedup12` | |ρ|≥0.95 중복 쌍에서 permutation importance 높은 쪽만 남김 |
| `top10` | permutation importance 상위 10 |
| `no_ccct` | 25 − ccct (라벨 누설 의심 제거) |
| `no_ccct_dedup` | dedup12 − ccct |
| `deploy4` | 보드 측정 가능: dcirohm, deltat103840, cvct, ccct |
| `deploy3` | 위에서 ccct 제외 |


In [ ]:
FEATURE_SETS = {
    "all25":         ALL_FEATURES,
    "dedup12":       ["ccct", "variance", "dcirohm", "mean3839", "mean4041", "range", "ctr", "dcir_soc10ohm",
                      "dcir_soc80ohm", "stdev", "deltat023840", "cvct"],
    "top10":         ["ccct", "variance", "dcirohm", "mean3839", "deltat103840", "ctr", "dcir_soc10ohm",
                      "range", "deltat023840", "mean4041"],
    "no_ccct":       [f for f in ALL_FEATURES if f != "ccct"],
    "no_ccct_dedup": ["variance", "dcirohm", "mean3839", "mean4041", "range", "ctr", "dcir_soc10ohm",
                      "dcir_soc80ohm", "stdev", "deltat023840", "cvct", "deltat103840"],
    "deploy4":       ["dcirohm", "deltat103840", "cvct", "ccct"],
    "deploy3":       ["dcirohm", "deltat103840", "cvct"],
}
for name, fs in FEATURE_SETS.items():
    if name == "all25": continue
    run_exp(f"E: {name}", {}, feats=fs, group="E")

# §7. F — 구조 (hidden 크기)

16-8이 FPGA 합의지만, 8-4로 줄여도 성능이 같으면 자원 절반이고, 32-16이 확 좋으면 협의 근거가 된다.


In [ ]:
for h in [(8, 4), (32, 16), (16, 16, 8)]:
    run_exp(f"F: hidden {h}", {"hidden": h}, group="F")
print("params:", {str(h): count_params(MLP(25, h)) for h in [(8, 4), (16, 8), (32, 16), (16, 16, 8)]})

# §8. G — Seed 앙상블

같은 fold에서 seed만 다른 MLP 5개를 평균. 분산을 줄여주지만 FPGA에선 5배 비용. (참고: 위 `OOF_RMSE(seed-avg)` 열이 이미 3-seed 앙상블 효과를 보여줌)


In [ ]:
def ensemble_cv(cfg, feats=None, n_models=5):
    fi = fidx(feats) if feats else list(range(len(ALL_FEATURES)))
    oof = np.zeros(len(y_dev)); fold_r = []
    for k, (tr, va) in enumerate(FOLDS, 1):
        ps = []
        for s in range(n_models):
            set_seed(1000 * s + k)
            m, pp, _, _ = train_mlp(X_dev[tr][:, fi], y_dev[tr], X_dev[va][:, fi], y_dev[va], g_dev[tr], r_dev[tr], cfg)
            ps.append(predict(m, pp, X_dev[va][:, fi]))
        p = np.mean(ps, axis=0); oof[va] = p; fold_r.append(rmse(y_dev[va], p))
    return np.mean(fold_r), np.std(fold_r), rmse(y_dev, oof)

m_, s_, o_ = ensemble_cv({}, n_models=5)
RESULTS.append({"group": "G", "exp": "G: 5-seed ensemble (all25)", "n_feat": 25, "fold_RMSE_mean": m_, "fold_RMSE_std": s_,
                "OOF_RMSE(seed-avg)": o_, "OOF_MAE": np.nan, "OOF_R2": np.nan, "OOF_ErrRate_%": np.nan, "best_ep": np.nan, "sec": np.nan})
print(f"[ G] 5-seed ensemble  fold RMSE {m_:.3f} ± {s_:.3f}   OOF {o_:.3f}")

# §9. H — QAT (양자화 인지 학습)

forward에서 weight를 int8로 반올림해 쓰고 backward는 그대로 통과(straight-through). 학습 중에 양자화 오차를 미리 겪어서 int8 변환 후 손실이 줄어든다.
eval 노트북에선 post-training int8 손실이 이미 0.00이라 큰 의미는 없지만, feature를 줄이거나 activation도 양자화하면 필요해질 수 있어 준비해 둔다.


In [ ]:
run_exp("H: QAT int8 weight", {"qat": True}, group="H")

# §10. 실험 결과 종합

In [ ]:
res = results_df()
base_rmse = res.loc["baseline (all25, default)", "fold_RMSE_mean"]
res["Δ vs base"] = res["fold_RMSE_mean"] - base_rmse
res["Δ%"] = res["Δ vs base"] / base_rmse * 100
pd.set_option("display.width", 200)
display(res[["group", "n_feat", "fold_RMSE_mean", "fold_RMSE_std", "OOF_RMSE(seed-avg)", "OOF_MAE", "OOF_R2", "Δ vs base", "Δ%", "sec"]].round(3))

# ── 그래프: 실험별 fold RMSE (기준선 = 빨간 점선) ──
GROUP_C = {"0": "black", "A": "#1f77b4", "B": "#ff7f0e", "C": "#2ca02c", "D": "#d62728", "E": "#9467bd", "F": "#8c564b", "G": "#e377c2", "H": "#17becf"}
r = res.sort_values("fold_RMSE_mean", ascending=False)
fig, ax = plt.subplots(figsize=(10, 0.36 * len(r) + 1))
ax.barh(r.index, r["fold_RMSE_mean"], xerr=r["fold_RMSE_std"], color=[GROUP_C[g] for g in r["group"]], capsize=3, alpha=.85)
ax.axvline(base_rmse, color="red", ls="--", lw=1.2, label=f"baseline {base_rmse:.3f}")
for i, (n, v) in enumerate(zip(r.index, r["fold_RMSE_mean"])): ax.text(v + 0.02, i, f"{v:.3f}", va="center", fontsize=8)
ax.set_xlabel("5-fold RMSE (%SOH), mean ± std over folds×seeds — lower is better"); ax.set_title("MLP improvement experiments")
ax.legend(); plt.tight_layout(); plt.show()

## 10-1. 어느 fold가 어려운가 — 실험별 fold RMSE heatmap

기준선에서 fold 4(4.5V 셀 포함)가 유독 나빴다. 개선 기법이 **그 fold를 살리는지**, 아니면 쉬운 fold만 더 좋아지는지 본다.


In [ ]:
fold_tbl = {}
for name, oof in OOF_STORE.items():
    p = oof.mean(axis=0)
    fold_tbl[name] = [rmse(y_dev[va], p[va]) for _, va in FOLDS]
fold_tbl = pd.DataFrame(fold_tbl, index=[f"fold{k}" for k in range(1, 6)]).T
fig, ax = plt.subplots(figsize=(7, 0.36 * len(fold_tbl) + 1))
im = ax.imshow(fold_tbl.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(5)); ax.set_xticklabels(fold_tbl.columns); ax.set_yticks(range(len(fold_tbl))); ax.set_yticklabels(fold_tbl.index, fontsize=8)
for i in range(len(fold_tbl)):
    for j in range(5): ax.text(j, i, f"{fold_tbl.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.grid(False); ax.set_title("OOF RMSE per fold (seed-avg prediction)"); plt.colorbar(im, fraction=.03); plt.tight_layout(); plt.show()
print("fold별 검증 셀 조건:")
for k, (_, va) in enumerate(FOLDS, 1): print(f"  fold{k}: {sorted(set(df.iloc[dev_idx].iloc[va]['cond']))}")

# §11. 최종 config 조합

**규칙**: 단독으로 기준선보다 좋았고 fold std를 키우지 않은 것만 합친다. 개별 효과는 더해지지 않을 수 있으니 조합을 다시 검증한다.

1차 실행 결과로 고른 조합 (추론에 영향 없는 것만):
- `scheduler="cosine"` (patience 300) — 단독 최고 (0.68)
- `interp_aug=64` — 셀 내 보간 증강 (0.72)

빼기로 한 것: noise·huber·dropout·wd↑ (효과 없음/악화), lr 1e-3 (cosine과 중복), QAT (지금은 불필요).
자동 선택 코드(아래)는 참고용 — 여러 개를 다 합치면 오히려 0.67로 cosine 단독과 차이 없었다.


In [ ]:
# ── 참고: 자동 후보 (Δ<0 이고 FPGA 추론 무관인 A,B,C,H) ──
train_only = res[res["group"].isin(["A", "B", "C", "H"]) & (res["Δ vs base"] < 0)].sort_values("fold_RMSE_mean")
print("추론 무관 개선 후보 (좋은 순):"); display(train_only[["fold_RMSE_mean", "fold_RMSE_std", "Δ vs base"]].round(3))

# ★ 최종 학습 config (사람이 결정). 결과가 바뀌면 여기만 수정.
IMPROVED_CFG = {"scheduler": "cosine", "patience": 300, "interp_aug": 64}
print("IMPROVED_CFG =", IMPROVED_CFG)

## 11-1. 조합 검증 — 학습 config × feature 세트 × hidden

E 그룹에서 feature 축소 효과가 컸으니, 개선 config를 세트별로 다시 돌린다. hidden 8-4도 같이 (F 그룹에서 16-8보다 좋았음).


In [ ]:
COMBOS = [
    ("FINAL: all25 | cosine+interp64",          IMPROVED_CFG,                        "all25"),
    ("FINAL: top10 | cosine",                   {"scheduler": "cosine", "patience": 300}, "top10"),
    ("FINAL: top10 | cosine+interp64",          IMPROVED_CFG,                        "top10"),
    ("FINAL: top10 | cosine+interp64 + h(8,4)", {**IMPROVED_CFG, "hidden": (8, 4)},  "top10"),
    ("FINAL: dedup12 | cosine+interp64",        IMPROVED_CFG,                        "dedup12"),
    ("FINAL: deploy4 | cosine+interp64",        IMPROVED_CFG,                        "deploy4"),
]
for name, cfg, fs in COMBOS:
    run_exp(name, cfg, feats=FEATURE_SETS[fs], group="Z")

res = results_df(); res["Δ vs base"] = res["fold_RMSE_mean"] - base_rmse
cmp = res[res.index.str.startswith(("baseline", "E:", "FINAL"))][["n_feat", "fold_RMSE_mean", "fold_RMSE_std", "OOF_RMSE(seed-avg)", "Δ vs base"]].round(3)
display(cmp)

# 기본 vs 개선, feature 세트별 나란히
pairs = [("all25",   "baseline (all25, default)", "FINAL: all25 | cosine+interp64"),
         ("top10",   "E: top10",                  "FINAL: top10 | cosine+interp64"),
         ("dedup12", "E: dedup12",                "FINAL: dedup12 | cosine+interp64"),
         ("deploy4", "E: deploy4",                "FINAL: deploy4 | cosine+interp64")]
fig, ax = plt.subplots(figsize=(9, 4.2)); w = 0.38; xs = np.arange(len(pairs))
b = [res.loc[p[1], "fold_RMSE_mean"] for p in pairs]; i_ = [res.loc[p[2], "fold_RMSE_mean"] for p in pairs]
bs = [res.loc[p[1], "fold_RMSE_std"] for p in pairs]; is_ = [res.loc[p[2], "fold_RMSE_std"] for p in pairs]
ax.bar(xs - w/2, b, w, yerr=bs, capsize=3, label="default cfg", color="#9e9e9e")
ax.bar(xs + w/2, i_, w, yerr=is_, capsize=3, label="cosine + interp aug", color="#1f77b4")
for x, v1, v2 in zip(xs, b, i_): ax.text(x - w/2, v1, f"{v1:.2f}", ha="center", va="bottom", fontsize=8); ax.text(x + w/2, v2, f"{v2:.2f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(xs); ax.set_xticklabels([f"{p[0]}\n({res.loc[p[1],'n_feat']} feat)" for p in pairs]); ax.set_ylabel("5-fold RMSE (%SOH)")
ax.set_title("Default vs improved training config, per feature set"); ax.legend(); plt.tight_layout(); plt.show()

# §12. Holdout 10-seed 검증

CV로 고른 설정이 holdout(5셀)에서도 좋은지, seed 10개로 평균±표준편차. 세 가지 비교:
1. `default / all25` — eval 노트북 설정
2. `improved / all25` — 학습 config만 바꿈 (feature 그대로)
3. `improved / FINAL_FEATS` — feature 축소까지

`FINAL_FEATS`는 팀 결정에 따라 바꾼다 (기본 top10).


In [ ]:
FINAL_FEATS = "top10"          # ← 팀 결정 후 변경: "all25" / "dedup12" / "deploy4"
fi_all = list(range(len(ALL_FEATURES))); fi = fidx(FEATURE_SETS[FINAL_FEATS])

def holdout_runs(cfg, fi, seeds=range(10)):
    out, preds = [], []
    for s in seeds:
        gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=1000 + s)   # early-stopping용 val 셀도 seed마다 바꿈
        tr2, va2 = next(gss2.split(X_dev, y_dev, groups=g_dev))
        set_seed(s)
        m, pp, _, _ = train_mlp(X_dev[tr2][:, fi], y_dev[tr2], X_dev[va2][:, fi], y_dev[va2], g_dev[tr2], r_dev[tr2], cfg)
        p = predict(m, pp, X_test[:, fi]); preds.append(p); out.append(metrics(y_test, p))
    return pd.DataFrame(out), np.array(preds)

runs = {
    "default / all25":            holdout_runs({}, fi_all),
    "improved / all25":           holdout_runs(IMPROVED_CFG, fi_all),
    f"improved / {FINAL_FEATS}":  holdout_runs(IMPROVED_CFG, fi),
}
summary = pd.concat({k: v[0].agg(["mean", "std"]).T for k, v in runs.items()}, axis=1)
print("Holdout, 10 seeds:"); display(summary.round(3))

names = list(runs); cols = ["#9e9e9e", "#6baed6", "#1f77b4"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
axes[0].boxplot([runs[n][0]["RMSE"] for n in names], labels=names); axes[0].set_ylabel("holdout RMSE (%)"); axes[0].set_title("RMSE over 10 seeds")
for i, n in enumerate(names):
    v = runs[n][0]["RMSE"]; axes[0].scatter(np.full(len(v), i + 1) + np.random.uniform(-.08, .08, len(v)), v, s=14, color="k", zorder=3)
axes[1].boxplot([runs[n][0]["MaxAbsErr"] for n in names], labels=names); axes[1].set_title("Max abs error over 10 seeds")
lo, hi = y_test.min() - 1, y_test.max() + 1
for n, c in zip(names, cols):
    pm = runs[n][1].mean(0); axes[2].scatter(y_test, pm, s=26, alpha=.7, color=c, label=f"{n}  RMSE {rmse(y_test, pm):.2f}")
axes[2].plot([lo, hi], [lo, hi], "--", color="gray"); axes[2].set_xlabel("actual SOH (%)"); axes[2].set_ylabel("predicted (10-seed mean)"); axes[2].set_title("Holdout: predicted vs actual"); axes[2].legend(fontsize=8)
for ax in axes[:2]: ax.tick_params(axis="x", labelsize=8)
plt.tight_layout(); plt.show()

# §13. 최종 모델 학습 + Export

개선 config로 dev 전체 학습 → int8 → 펌웨어/FPGA 파일. `preproc`에 log 변환이 포함되면 `log_cols`가 같이 저장되니 펌웨어에서 그 열만 log 먼저 취해야 한다.


In [ ]:
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42 + 99)
tr2, va2 = next(gss2.split(X_dev, y_dev, groups=g_dev))
set_seed(42)
final, pp, hist, best_ep = train_mlp(X_dev[tr2][:, fi], y_dev[tr2], X_dev[va2][:, fi], y_dev[va2], g_dev[tr2], r_dev[tr2], IMPROVED_CFG, verbose=True)
p_test = predict(final, pp, X_test[:, fi])
print(f"\nfinal ({FINAL_FEATS}) holdout: {metrics(y_test, p_test)}")

# loss curve
plt.figure(figsize=(8, 3.8)); plt.plot(hist["train"], label="train"); plt.plot(hist["val"], label="val"); plt.axvline(best_ep, color="gray", ls=":")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("loss (scaled)"); plt.title(f"Final improved MLP — best epoch {best_ep}"); plt.legend(); plt.show()

# ── export ──
feats = FEATURE_SETS[FINAL_FEATS]
export = {"features": feats, "log_cols": [feats[i] for i, gi in enumerate(fi) if gi in IMPROVED_CFG.get("log_cols", [])],
          **pp.params(), "y_offset": Y_OFFSET, "y_scale": Y_SCALE, "hidden": list(IMPROVED_CFG.get("hidden", (16, 8))), "cfg": {k: (list(v) if isinstance(v, tuple) else v) for k, v in {**BASE_CFG, **IMPROVED_CFG}.items()}}
json.dump(export, open("scaler_params_improved.json", "w"), indent=2)

def q8(w):
    s = np.max(np.abs(w)) / 127.0 if np.max(np.abs(w)) > 0 else 1.0
    return np.clip(np.round(w / s), -127, 127).astype(np.int8), s
q_state, q_meta = {}, {}
for k, v in final.state_dict().items():
    w = v.numpy()
    if k.endswith("weight"):
        q, s = q8(w); q_state[k] = torch.tensor(q.astype(np.float32) * s); q_meta[k] = {"scale": float(s), "int8": q.tolist()}
    else:
        q_state[k] = v.clone(); q_meta[k] = {"float": w.tolist()}
qm = MLP(len(fi), tuple(IMPROVED_CFG.get("hidden", (16, 8)))); qm.load_state_dict(q_state)
p_q = predict(qm, pp, X_test[:, fi])
print(f"float RMSE {rmse(y_test, p_test):.3f}  int8-w RMSE {rmse(y_test, p_q):.3f}  max|Δ| {np.max(np.abs(p_test - p_q)):.3f}")
json.dump(q_meta, open("mlp_weights_int8_improved.json", "w"))
json.dump({k: v.numpy().tolist() for k, v in final.state_dict().items()}, open("mlp_weights_float_improved.json", "w"))
gv = pd.DataFrame(pp.transform(X_test[:, fi]), columns=[f"x_{f}" for f in feats]); gv["soh_actual"] = y_test; gv["soh_pred_float"] = p_test; gv["soh_pred_int8w"] = p_q
gv.to_csv("golden_test_vectors_improved.csv", index=False)
res.to_csv("improve_experiments.csv")
torch.save(final.state_dict(), "best_mlp_improved.pth")
print("saved: scaler_params_improved.json, mlp_weights_{float,int8}_improved.json, golden_test_vectors_improved.csv, improve_experiments.csv")

# §14. 해석 가이드

**1차 실행 결과 요약**: 기준선 0.83 → cosine+보간증강 0.56 (all25) → +top10 0.35 → +hidden 8-4 0.33. 오차는 거의 fold 4(4.35V/4.5V 과충전 셀)에서 나오고, feature 축소가 그 fold를 가장 많이 살린다.


- **Δ가 −0.05 이하**면 의미 있는 개선. fold std(±0.4~0.5)에 비해 작으면 노이즈일 수 있으니 §12 holdout 10-seed로 재확인.
- **A(증강)** 가 효과 있으면 → 데이터 부족이 병목. 나머지 36셀 병합이 가장 큰 개선이 될 것.
- **E(feature 축소)** 에서 `no_ccct`가 크게 나빠지면 → 지금 성능의 상당 부분이 ccct(라벨 누설 성격)에서 온 것. 팀 회의 안건.
- **F** 에서 8-4가 16-8과 같다면 → FPGA 자원 절반으로 협의 가능.
- **D(log/robust)** 는 좋아도 펌웨어 정규화가 복잡해지니 이득이 클 때만 채택.
